# $F_A(Q^2)$ fractional uncertainty: paper priors and our fits

This notebook checks the paper's quoted precision at $Q^2=0.50\;\mathrm{GeV}^2$ and then applies the same definition to our post-fit chains. The paper values are reproduced directly from the full correlated $k_{\max}=6$ coefficient covariances used by this analysis.

We define the fractional uncertainty as
$$\delta_F(Q^2)=\frac{\sigma[F_A(Q^2)]}{|\mathbb{E}[F_A(Q^2)]|}. $$
For a Gaussian coefficient prior, this is evaluated analytically with its full covariance. For a posterior chain, it is evaluated from the sampled $F_A$ distribution.

**Spline-factorization correction.** Every posterior quantity below is computed with the importance weights $w_k=\exp(+\Delta\chi^2_{\rm data}(\eta^{(k)})/2)$ that correct PROfit's multiplicative combination of the one-dimensional PCA splines to the exact z-expansion response (`python/scripts/spline_reweighting.py`, grids from `12_spline_factorization_validation.ipynb`, demonstration in `13_spline_reweighting_comparison.ipynb`). The weights are applied inside `postfit_physical_parameters.load_fit` and propagate to the summary tables, covariances, credible bands and corner plots; the effective sample size after reweighting is printed when each chain is loaded. The correction is negligible for the LQCD-constrained and Gaussian MINERvA priors and matters only for the uniform-prior fits.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

start = Path.cwd().resolve()
NOTEBOOK_DIR = next(
    (path / 'python' / 'scripts' for path in (start, *start.parents)
     if (path / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('Could not locate python/scripts/postfit_physical_parameters.py')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from postfit_physical_parameters import (
    SPECS, _fa_curves, _zexp_transform, load_fit,
)
from spline_reweighting import describe_ess, weighted_mean_std

## Controls

The nominal NuWro-based fit is listed first. Open-data and Asimov fits are included for a direct cross-check and can be removed from `SUITES` if only the nominal result is wanted.

In [ ]:
Q2_REFERENCE = 0.50  # GeV^2
FIT_KEYS = ('minerva_k6', 'lqcd_k6', 'minerva_lqcd_k6')
SUITES = {
    'Nominal (NuWro)': 'nuwro_fit_results',
    'Open data': 'opendata_fit_results',
    'Asimov': 'asimov_fit_results',
}
BURN_IN = 0
THIN = 1
MAX_SAMPLES = 100_000
Q2 = np.linspace(0.0, 2.0, 301)

In [ ]:
specs = {spec.key: spec for spec in SPECS}

def z_basis(prior, q2, n_coefficients):
    q2 = np.atleast_1d(q2).astype(float)
    z = (
        np.sqrt(prior.t_cut_gev2 + q2)
        - np.sqrt(prior.t_cut_gev2 - prior.t0_gev2)
    ) / (
        np.sqrt(prior.t_cut_gev2 + q2)
        + np.sqrt(prior.t_cut_gev2 - prior.t0_gev2)
    )
    return np.vander(z, N=n_coefficients, increasing=True)


def prior_fa_statistics(spec, q2):
    central, coefficient_transform = _zexp_transform(spec.prior)
    covariance = coefficient_transform @ coefficient_transform.T
    basis = z_basis(spec.prior, q2, len(central))
    mean = basis @ central
    variance = np.einsum('ij,jk,ik->i', basis, covariance, basis)
    sigma = np.sqrt(np.clip(variance, 0.0, None))
    return mean, sigma, sigma / np.abs(mean)


def posterior_fa_statistics(result, q2, max_samples=MAX_SAMPLES):
    # Importance-weighted moments: the weights correct the spline factorization.
    curves, weights = _fa_curves(
        result, np.atleast_1d(q2), use_prior=False, max_samples=max_samples,
        return_weights=True,
    )
    mean, sigma = weighted_mean_std(curves, weights)
    return mean, sigma, sigma / np.abs(mean)

## 1. Reproduce the paper statement

The table uses the full covariance, including correlations among the independent $z$-expansion coefficients and the coefficients derived from the sum rules. Agreement should be assessed against the paper's rounded values of 7% and 2%.

In [ ]:
paper_quotes = {'minerva_k6': 7.0, 'minerva_lqcd_k6': 2.0}
rows = []
for key in FIT_KEYS:
    mean, sigma, fractional = prior_fa_statistics(specs[key], Q2_REFERENCE)
    calculated = 100 * fractional[0]
    rows.append({
        'Prior': specs[key].title.replace('$', ''),
        'F_A(0.50)': mean[0],
        'sigma[F_A]': sigma[0],
        'Calculated fractional uncertainty [%]': calculated,
        'Paper quote [%]': paper_quotes.get(key, np.nan),
    })
paper_check = pd.DataFrame(rows).set_index('Prior')
display(paper_check.style.format({
    'F_A(0.50)': '{:.6f}',
    'sigma[F_A]': '{:.6f}',
    'Calculated fractional uncertainty [%]': '{:.2f}',
    'Paper quote [%]': lambda value: '—' if pd.isna(value) else f'{value:.0f}',
}))

# for key, quoted in paper_quotes.items():
#     _, _, fractional = prior_fa_statistics(specs[key], Q2_REFERENCE)
#     assert np.isclose(100 * fractional[0], quoted, atol=0.5), (
#         key, 100 * fractional[0], quoted
#     )
# print('The encoded priors reproduce both rounded paper claims within 0.5 percentage points.')

## 2. Compare our post-fit precision at the same point

Each posterior is compared with its own paper prior. A ratio below one means our fit reduces the uncertainty. This is a precision comparison only; shifts in the central value are reported separately and should not be interpreted as improved precision.

In [ ]:
results = {}
for suite_label, suite in SUITES.items():
    results[suite_label] = {}
    for key in FIT_KEYS:
        result = load_fit(specs[key], suite, burn_in=BURN_IN, thin=THIN)
        if result is None:
            raise FileNotFoundError(
                f'No unique PROfile ROOT file for {key!r} in {suite!r}'
            )
        results[suite_label][key] = result
        print(f'{suite_label:16s} | {key:18s} | {len(result["samples"]):,} samples | '
              + describe_ess(result['ess'], len(result['samples'])))

In [ ]:
comparison_rows = []
for suite_label, suite_results in results.items():
    for key, result in suite_results.items():
        prior_mean, _, prior_fractional = prior_fa_statistics(
            specs[key], Q2_REFERENCE
        )
        post_mean, _, post_fractional = posterior_fa_statistics(
            result, Q2_REFERENCE
        )
        comparison_rows.append({
            'Dataset': suite_label,
            'Prior': specs[key].title.replace('$', ''),
            'Prior uncertainty [%]': 100 * prior_fractional[0],
            'Posterior uncertainty [%]': 100 * post_fractional[0],
            'Posterior / prior uncertainty': post_fractional[0] / prior_fractional[0],
            'Prior F_A': prior_mean[0],
            'Posterior mean F_A': post_mean[0],
            'Posterior mean shift [% of prior F_A]': (
                100 * (post_mean[0] - prior_mean[0]) / abs(prior_mean[0])
            ),
        })
comparison = pd.DataFrame(comparison_rows).set_index(['Dataset', 'Prior'])
display(comparison.style.format({
    'Prior uncertainty [%]': '{:.2f}',
    'Posterior uncertainty [%]': '{:.2f}',
    'Posterior / prior uncertainty': '{:.3f}',
    'Prior F_A': '{:.5f}',
    'Posterior mean F_A': '{:.5f}',
    'Posterior mean shift [% of prior F_A]': '{:+.2f}',
}))

## 3. Fractional uncertainty versus $Q^2$

Solid curves show our posterior precision and dashed curves show the corresponding paper prior. The vertical line marks the paper's quoted comparison point.

In [ ]:
colors = {
    'minerva_k6': '#56B4E9',
    'lqcd_k6': '#009E73',
    'minerva_lqcd_k6': '#CC79A7',
}
short_labels = {
    'minerva_k6': r'MINERvA (2026), $k_{\max}=6$',
    'lqcd_k6': r'LQCD (2026), $k_{\max}=6$',
    'minerva_lqcd_k6': r'MINERvA + LQCD (2026), $k_{\max}=6$',
}

# Evaluate once so the combined and suite-specific figures are identical.
prior_curves = {}
posterior_curves = {}
for key in FIT_KEYS:
    _, _, prior_curves[key] = prior_fa_statistics(specs[key], Q2)
for suite_label, suite_results in results.items():
    posterior_curves[suite_label] = {}
    for key in FIT_KEYS:
        _, _, posterior_curves[suite_label][key] = posterior_fa_statistics(
            suite_results[key], Q2
        )

def style_uncertainty_axis(ax, suite_label):
    for key in FIT_KEYS:
        ax.plot(
            Q2, 100 * prior_curves[key], color=colors[key],
            ls=(0, (4, 2.4)), lw=1.45, alpha=.85,
        )
        ax.plot(
            Q2, 100 * posterior_curves[suite_label][key],
            color=colors[key], lw=2.25, label=short_labels[key],
        )
    ax.set_xlim(Q2[0], Q2[-1])
    ax.set_ylim(bottom=0)
    ax.set_ylabel(r'Fractional uncertainty on $F_A$ [\%]')
    ax.grid(which='major', color='#9AA4B2', alpha=.22, linewidth=.7)
    ax.grid(which='minor', axis='x', color='#9AA4B2', alpha=.10, linewidth=.5)
    ax.tick_params(direction='in', top=True, right=True)

def save_publication_figure(figure, stem):
    stem.parent.mkdir(parents=True, exist_ok=True)
    for extension in ('pdf',):
        path = stem.with_suffix('.' + extension)
        kwargs = dict(bbox_inches='tight', pad_inches=.03, facecolor='white')
        if extension == 'png':
            kwargs['dpi'] = 600
        figure.savefig(path, **kwargs)
        print(f'Saved {path}')

style_handles = [
    Line2D([], [], color='0.25', lw=1.45, ls=(0, (4, 2.4)), label='Prior'),
    Line2D([], [], color='0.25', lw=2.25, label='Posterior'),
] + [
    Line2D([], [], color=colors[key], lw=2.0, label=short_labels[key])
    for key in ('lqcd_k6', 'minerva_k6', 'minerva_lqcd_k6')
]

# Match fa_postfit_summary: one 7.1-inch-wide figure per dataset,
# colorblind-safe source colors, inward ticks, light grid, and frameless legend.
for suite_label, suite in SUITES.items():
    figure, axis = plt.subplots(figsize=(7.1, 4.8))
    style_uncertainty_axis(axis, suite_label)
    axis.set_xlabel(r'$Q^2$ [GeV$^2$]')
    axis.legend(
        handles=style_handles, frameon=False, fontsize=9.5, ncol=1,
        loc='upper left', handlelength=2.8, columnspacing=1.2,
    )
    figure.tight_layout()
    save_publication_figure(
        figure, NOTEBOOK_DIR.parents[1] / 'figs' / suite / 'fa_fractional_uncertainty'
    )
    display(figure)
    plt.close(figure)

## Reading the result

At $Q^2=0.50\;\mathrm{GeV}^2$, the prior check should return approximately **6.82%** for MINERvA-only and **1.62%** for MINERvA+LQCD, reproducing the paper's rounded 7% and 2%. The posterior table is the direct answer for our fits. Compare `Posterior uncertainty [%]` with `Prior uncertainty [%]`; the ratio column quantifies the constraint added by our data. The final two columns expose central-value movement, which is conceptually separate from uncertainty reduction.

In [ ]:
# Change this value, then rerun the cell. Units are GeV^2.
Q2_VALUE = 0.50

if not np.isfinite(Q2_VALUE) or Q2_VALUE < 0:
    raise ValueError('Q2_VALUE must be a finite, non-negative number in GeV^2.')

rows = []
for key in FIT_KEYS:
    _, _, prior_fractional = prior_fa_statistics(specs[key], Q2_VALUE)
    row = {
        'Parameterization': specs[key].title.replace('$', ''),
        'Paper prior [%]': 100 * prior_fractional[0],
    }
    for suite_label, suite_results in results.items():
        _, _, post_fractional = posterior_fa_statistics(
            suite_results[key], Q2_VALUE
        )
        row[f'{suite_label} posterior [%]'] = 100 * post_fractional[0]
    rows.append(row)

uncertainty_at_q2 = pd.DataFrame(rows).set_index('Parameterization')
display(
    uncertainty_at_q2.style
    .format('{:.2f}')
    .set_caption(
        rf'Fractional uncertainty $\sigma(F_A)/|\langle F_A\rangle|$ '
        rf'at $Q^2={Q2_VALUE:g}\;\mathrm{{GeV}}^2$'
    )
)